[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C4/notebooks/gradienty.ipynb)

# Tři gradienty v jednom gradientním sestupu

Gradientní sestup je jeden řádek, $\mathbf x \leftarrow \mathbf x - \alpha\nabla f(\mathbf x)$.
Celá otázka je, **odkud vzít** $\nabla f$. Funkce z tabule je

$$f(x_1,x_2)=(x_1-1)^2+(x_1x_2-2)^2,$$

minimum má v $(1;\ 2)$. Na tabuli vyšel gradient v bodě $(1;\ 1)$ roven
$(-2;\ -2)$ a dva kroky s $\alpha = 0{,}1$ vedly do $(1{,}2;\ 1{,}2)$
a $(1{,}2944;\ 1{,}3344)$.

Doplňte **tři místa `# TODO`**, tři zaměnitelné funkce `grad(x)`:

| cesta | co potřebuje | funkce |
|---|---|---|
| numerický (*finite differences*) | jen umět spočítat f | `grad_numericky` |
| analytický (ručně) | vzorec z tabule | `grad_analyticky` |
| automatický (*autograd*) | f napsanou v PyTorchi | `grad_autograd` |

---

*Čísla, se kterými se notebook porovnává, počítají skripty [`kod/cisla.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C4/kod/cisla.py), [`kod/cena_gradientu.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C4/kod/cena_gradientu.py).*

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

def f(x):
    """Funguje pro pole NumPy i pro tenzor PyTorch."""
    return (x[0] - 1) ** 2 + (x[0] * x[1] - 2) ** 2

def gd(grad, x0, alpha, kroku=20000, tol=1e-6):
    """Gradientní sestup; vrátí celou cestu (řádek = jeden bod)."""
    x = np.array(x0, dtype=float)
    cesta = [x]
    for _ in range(kroku):
        g = grad(x)
        if np.linalg.norm(g) < tol:  # gradient skoro nulový: jsme v minimu
            break
        x = x - alpha * g
        cesta.append(x)
    return np.array(cesta)

## Část 1 — tři gradienty *(díry 1–3)*

In [ ]:
def grad_numericky(x, fce=f, h=1e-5):
    """Centrální diference: pro každou souřadnici dvě vyhodnocení fce."""
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    for i in range(len(x)):
        e = np.zeros_like(x)
        e[i] = h                     # posun jen v i-té souřadnici
        # TODO 1: centrální diference  (f(x + e) - f(x - e)) / 2h
        g[i] = ...
    return g

In [ ]:
def grad_analyticky(x):
    """Vzorec odvozený na tabuli (platí jen pro tuhle f se dvěma proměnnými)."""
    x1, x2 = x
    # TODO 2: parciální derivace podle x1 a podle x2
    d1 = ...
    d2 = ...
    return np.array([d1, d2])

In [ ]:
def grad_autograd(x, fce=f):
    """PyTorch si při výpočtu fce postaví výpočetní graf a projde ho pozpátku."""
    # TODO 3: tenzor s requires_grad=True, zavolat backward() a vrátit .grad
    xt = torch.tensor(x, dtype=torch.float64)  # ... doplnit requires_grad=True
    hodnota = fce(xt)
    ...
    return ...

In [ ]:
gradienty = {"numerický": grad_numericky, "analytický": grad_analyticky,
             "autograd": grad_autograd}
cesty = {}
for jmeno, grad in gradienty.items():
    cesta = gd(grad, [1.0, 1.0], alpha=0.1)
    cesty[jmeno] = cesta
    print(f"{jmeno:>10}: grad(1; 1) = {np.round(grad([1.0, 1.0]), 6)}, "
          f"první krok {np.round(cesta[1], 6)}, {len(cesta) - 1} iterací, "
          f"konec {np.round(cesta[-1], 4)}")
    assert np.allclose(grad([1.0, 1.0]), [-2, -2], atol=1e-6), "gradient nesedí s tabulí"
    assert np.allclose(cesta[1], [1.2, 1.2], atol=1e-6), "první krok nesedí s tabulí"
    assert np.allclose(cesta[2], [1.2944, 1.3344], atol=1e-6), "druhý krok nesedí s tabulí"
    assert abs(len(cesta) - 1 - 340) <= 2 and np.allclose(cesta[-1], [1, 2], atol=1e-5)

x1, x2 = np.meshgrid(np.linspace(0.3, 2.2, 200), np.linspace(0.5, 2.6, 200))
plt.contour(x1, x2, f([x1, x2]), levels=np.geomspace(0.01, 10, 12), cmap="viridis")
for (jmeno, cesta), styl in zip(cesty.items(), ["-", "--", ":"]):
    plt.plot(cesta[:, 0], cesta[:, 1], styl, marker=".", label=jmeno)
plt.plot(1, 2, "k*", markersize=12)
plt.xlabel("x1"); plt.ylabel("x2"); plt.legend(); plt.gca().set_aspect("equal")
plt.show()

Všechny tři cesty leží přes sebe. Gradientnímu sestupu je jedno, odkud gradient
dostane. **Pokus:** spusťte `gd(grad_analyticky, [1.0, 1.0], alpha=0.2)` a porovnejte
s tím, co jste tipovali na tabuli.

## Část 2 — kolik stojí jeden gradient

Funkci zobecníme na $n$ proměnných (pro $n = 2$ je to přesně ta z tabule):

$$f(\mathbf x)=\sum_{i=1}^{n-1}\left[(x_i-1)^2+(x_i x_{i+1}-2)^2\right].$$

Numerický gradient a autograd dostanou jen novou funkci. Analytický by bylo
potřeba odvodit znovu.

In [ ]:
def f_n(x):
    return ((x[:-1] - 1) ** 2 + (x[:-1] * x[1:] - 2) ** 2).sum()

def cas(grad, x):
    """Nejkratší ze tří měření jednoho gradientu, v sekundách."""
    casy = []
    for _ in range(3):
        start = time.perf_counter()
        grad(x, fce=f_n)
        casy.append(time.perf_counter() - start)
    return min(casy)

rng = np.random.default_rng(0)
N = [10, 100, 1000, 10000]
cas_num, cas_auto = [], []
for n in N:
    x = rng.uniform(0, 2, n)
    rozdil = np.abs(grad_numericky(x, f_n) - grad_autograd(x, f_n)).max()
    cas_num.append(cas(grad_numericky, x))
    cas_auto.append(cas(grad_autograd, x))
    print(f"n = {n:>5}: numerický {cas_num[-1] * 1e3:8.2f} ms, autograd {cas_auto[-1] * 1e3:6.2f} ms, "
          f"max. rozdíl {rozdil:.1e}")
    assert rozdil < 1e-5, "numerický a autograd se rozcházejí"

plt.loglog(N, cas_num, "o-", label="numerický (2n vyhodnocení f)")
plt.loglog(N, cas_auto, "s-", label="autograd (tam a zpět)")
plt.xlabel("počet proměnných n"); plt.ylabel("čas jednoho gradientu [s]"); plt.legend()
plt.show()